# 🧠 Sprint 1: Real-World Agent Loop Pattern

This notebook demonstrates the **Architecture** we will use for Sprint 1.

**Goal**: Compare a "Basic" LLM implementation vs a "State Machine" implementation to see why we need the complex one.

**Prerequisites**:
1. `pip install -r ../requirements.txt`
2. Ensure `GROQ_API_KEY` is in your `.env` file.

---


In [6]:
import os
import json
from typing import List, Dict, Any, Optional
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from groq import Groq

# Load env
load_dotenv("../.env")
api_key = os.getenv("GROQ_API_KEY")
model_id = os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile")

if not api_key:
    print("❌ ERROR: GROQ_API_KEY not found in .env")
else:
    print(f"✅ Connected to Groq using model: {model_id}")
    client = Groq(api_key=api_key)

✅ Connected to Groq using model: llama-3.3-70b-versatile


## 1. The Real-World State Object

In production, we use **Pydantic** to enforce `types`. If the LLM tries to put a "maybe" string into a `budget` integer field, Pydantic will yell, and we can catch that error.

In [7]:
class VacationPlan(BaseModel):
    destination_candidates: List[str] = Field(description="List of destinations being considered", default_factory=list)
    budget_range: Optional[str] = Field(description="Budget range (e.g. '$2000-$3000')", default=None)
    travelers: int = Field(description="Number of travelers", default=1)
    dates: Optional[str] = Field(description="Travel dates", default=None)
    requirements: List[str] = Field(description="Hard requirements (e.g. 'pet friendly', 'beach')", default_factory=list)

current_state = VacationPlan()
print("Initial State:", current_state.model_dump_json(indent=2))

Initial State: {
  "destination_candidates": [],
  "budget_range": null,
  "travelers": 1,
  "dates": null,
  "requirements": []
}


## 2. Architecture Comparison

### Approach A: "The Chatbot" (Bad for Agents)
We just dump the state in the prompt and ask the LLM to "keep it in mind".
*Why it fails*: As conversation grows, the model forgets constraints or hallucinates updates without actually structured data changing.

In [8]:
def run_chatbot_turn(user_input: str, history: List[Dict]):
    history.append({"role": "user", "content": user_input})
    
    # Simple Prompt
    messages = [
        {"role": "system", "content": "You are a travel agent. Help the user select a destination to travel to."},
    ] + history
    
    response = client.chat.completions.create(
        model=model_id,
        messages=messages
    )
    
    reply = response.choices[0].message.content
    history.append({"role": "assistant", "content": reply})
    return reply

# Simulating a chat
history_a = []
print("👤 User: I want to go to Italy.")
reply = run_chatbot_turn("I want to go to Italy, looking to spend $2k max.", history_a)
print(f"🤖 Bot: {reply}")

👤 User: I want to go to Italy.
🤖 Bot: Italy is a wonderful destination. With a budget of $2,000, you can have a fantastic time exploring the country's rich history, art, architecture, and delicious food. Here are a few options to consider:

1. **Rome**: Explore the Colosseum, Vatican City, and indulge in delicious Italian cuisine. You can find affordable accommodations in the city center, and budget-friendly restaurants serving pasta and pizza. ($1,500 - $1,800 for 7-10 days)
2. **Florence**: Visit the birthplace of the Renaissance, see Michelangelo's David, and cross the Ponte Vecchio. Florence is a bit more expensive than Rome, but you can still find budget-friendly options. ($1,800 - $2,000 for 7-10 days)
3. **Cinque Terre**: Hike through the five colorful towns on the Ligurian coast, swim in the Mediterranean, and enjoy fresh seafood. This destination is a bit more off-the-beaten-path, but still accessible with your budget. ($1,500 - $1,800 for 7-10 days)
4. **Tuscany**: Experience

### Approach B: "The Agent" (Our Sprint 1 Goal)
We force the model to use a **Tool** to update the state *before* it replies.
This creates a clear separation between "Thinking/Updating" and "Talking".

In [9]:
# The Tool Definition (JSON Schema for the LLM)
update_res_tool = {
    "type": "function",
    "function": {
        "name": "update_plan",
        "description": "Update the travel plan with new information.",
        "parameters":  VacationPlan.model_json_schema()
    }
}

def run_agent_turn(user_input: str, state: VacationPlan, history: List[Dict]):
    history.append({"role": "user", "content": user_input})
    
    # 1. System Prompt injects the STATE
    system_prompt = f"""You are a travel planner. Help the user select a destination to travel to.
Current Plan Schema: {state.model_dump_json()}
Behavior: If the user provides new info, CALL 'update_plan'. Then reply to the user."""
    
    messages = [{"role": "system", "content": system_prompt}] + history
    
    # 2. Call LLM with Tools enabled
    response = client.chat.completions.create(
        model=model_id,
        messages=messages,
        tools=[update_res_tool],
        tool_choice="auto"
    )
    
    msg = response.choices[0].message
    
    # 3. Check for Tool Call
    if msg.tool_calls:
        for tool in msg.tool_calls:
            if tool.function.name == "update_plan":
                args = json.loads(tool.function.arguments)
                print(f"⚙️ EXTRACTED UPDATES: {args}")
                
                # 4. EXECUTE UPDATE (Merge Logic)
                # In a real app complexity, we'd do clever merging. 
                # Here, Pydantic handles the 'patch' if we structure it right.
                updated_data = state.model_dump()
                updated_data.update(args)
                
                # Validate with Pydantic
                new_state = VacationPlan(**updated_data)
                
                # 5. Add Result to History
                history.append(msg)
                history.append({
                    "role": "tool",
                    "tool_call_id": tool.id,
                    "content": "Plan updated successfully. Now respond to the user confirming the update naturally."
                })
                
                # 6. SECOND LLM CALL (The "Response" Step)
                # The LLM now sees: User Request -> Tool Call -> Tool Output
                # It can generate a response like "I've updated your budget to $2000."
                messages_2 = [{"role": "system", "content": system_prompt}] + history
                response_2 = client.chat.completions.create(
                    model=model_id,
                    messages=messages_2
                )
                final_reply = response_2.choices[0].message.content
                history.append({"role": "assistant", "content": final_reply})
                
                return final_reply, new_state
    
    # If no tool call, just reply
    return msg.content, state


### Test the Agent
Notice how it extracts structured data *and* maintains the conversation.

In [10]:
history_b = []
state_b = VacationPlan()

print("---- TURN 1 ----")
reply, state_b = run_agent_turn("I want to go to Italy, budget is $2k.", state_b, history_b)
print(f"🤖 Agent: {reply}")
print(f"State after Turn 1: {state_b.model_dump_json(indent=2)}")

print("\n---- TURN 2 ----")
reply, state_b = run_agent_turn("Actually, make it 2 people and add Japan to the list.", state_b, history_b)
print(f"🤖 Agent: {reply}")
print(f"State after Turn 2: {state_b.model_dump_json(indent=2)}")

---- TURN 1 ----
⚙️ EXTRACTED UPDATES: {'budget_range': '$2000', 'dates': None, 'destination_candidates': ['Italy'], 'requirements': [], 'travelers': 1}
🤖 Agent: I've updated your travel plan. We're currently looking at Italy as a potential destination, and you have a budget of $2000. That's a great starting point! Italy has a lot to offer, from Rome's ancient history to Venice's canals and Florence's art scene. 

What time of year are you thinking of traveling? Are you flexible with your dates or do you have something specific in mind?
State after Turn 1: {
  "destination_candidates": [
    "Italy"
  ],
  "budget_range": "$2000",
  "travelers": 1,
  "dates": null,
  "requirements": []
}

---- TURN 2 ----
⚙️ EXTRACTED UPDATES: {'budget_range': '$2000', 'dates': None, 'destination_candidates': ['Italy', 'Japan'], 'requirements': [], 'travelers': 2}
🤖 Agent: I've updated your travel plan. We're now considering two exciting destinations: Italy and Japan. With a budget of $2000 for two peo

## Conclusion
With **Approach B**, we have a `state_b` object that is programmatically usable. 
- We can query `state_b.budget_range` to filter API results.
- We can show `state_b.destination_candidates` on a Map UI.
- **Approach A** gave us nothing but text.
 
**Crucially, the user gets a natural response**, while the system gets structured data updates in the background.

This is the core of our Sprint 1 architecture.